# ODI to Databricks Migration
**Source:** WC_MERCURY_BADGE_TS → WC_BADGE_DETAILS_D
**Schema:** PRXBI_DW_SEP → workspace.prxbi_dw / PRXBI_TS_SEP → workspace.prxbi_ts
**Description:** Incremental load of badge details from Mercury source into WC_BADGE_DETAILS_D dimension table.
Applies NOT_EXISTS detection strategy with IND_UPDATE flagging and MERGE into target.

In [ ]:
# Cell 1 —
Create ETL parameter widgets (SCEN_TASK_NO {1}-{6})
dbutils.widgets.text("ETL_JOB_TYPE", "")
dbutils.widgets.text("DATASOURCE_NUM_ID", "380")
dbutils.widgets.text("ETL_PROC_WID", "")
dbutils.widgets.text("ODI_SESS_NO", "")

## ETL Parameters
Temporary views for last/current extract times and ROW_WID seed from wc_etl_parameters.

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {2} / {4}: etl_last_extract_time
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT etl_last_extract_time
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {3} / {5}: etl_current_extract_time
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT etl_current_extract_time
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {6}: ROW_WID seed
CREATE OR REPLACE TEMPORARY VIEW v_etl_row_wid AS
SELECT ROW_WID
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
# Display ETL parameter values
display(spark.sql("""
SELECT (SELECT etl_last_extract_time
FROM v_etl_last_extract_time)    AS etl_last_extract_time,
    (SELECT etl_current_extract_time
FROM v_etl_current_extract_time) AS etl_current_extract_time,
    (SELECT ROW_WID
FROM v_etl_row_wid)              AS row_wid
"""))

## Staging Table (C$)
Drop, create, and populate the staging table from the Mercury Badge source.
The ODI MAX self-join dedup pattern has been replaced with ROW_NUMBER() (see Rule F.12).

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {30}:
Drop staging table
-- Converted:
DROP ... PURGE →
DROP TABLE IF EXISTS
DROP TABLE IF EXISTS workspace.prxbi_dw.c_0a7sucripsm1cg2656h955ou5qp_stg;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {40}: Create staging table
-- Converted: VARCHAR2 → STRING, NUMBER(5,0)/NUMBER(10,0) → BIGINT, TIMESTAMP(6) → TIMESTAMP, removed NOLOGGING
CREATE TABLE workspace.prxbi_dw.c_0a7sucripsm1cg2656h955ou5qp_stg (
    ID                              STRING,
    BADGELOCATION                   STRING,
    BADGETOKEN                      STRING,
    BADGEVERSION                    BIGINT,
    CONTACTEMAIL                    STRING,
    CONTACTFIRSTNAME                STRING,
    CONTACTJOBTITLE                 STRING,
    CONTACTLASTNAME                 STRING,
    CONTACTPERSONRXMASTERID         STRING,
    CREATEDBYREGISTRATIONTYPE       STRING,
    CREATEDBYTYPE                   STRING,
    CULTURE                         STRING,
    CUSTOMERTYPE                    STRING,
    EVENTEDITIONGBSCODE             STRING,
    ISBADGEUPDATE                   STRING,
    MARKETINGPREFERENCESPROMPTREQU  STRING,
    ORGANISATIONCITY                STRING,
    ORGANISATIONCOUNTRYCODE         STRING,
    ORGANISATIONDISPLAYNAME         STRING,
    ORGANISATIONRXMASTERID          STRING,
    ORGANISATIONSTATE               STRING,
    PARTICIPATINGORGANISATIONID     STRING,
    PRODUCTCODE                     STRING,
    QRCODECONTENT                   STRING,
    REGISTRATIONID                  STRING,
    STATUS                          BIGINT,
    SUPPORTSTAFFCOMPANYADDRESS      STRING,
    SUPPORTSTAFFCOMPANYNAME         STRING,
    SUPPORTSTAFFMOBILEPHONE         STRING,
    SUPPORTSTAFFREPORTSTO           STRING,
    SUPPORTSTAFFSTANDS              STRING,
    SUPPORTSTAFFUSERACCESS          STRING,
    VERSIONNUMBER                   BIGINT,
    MOBILEPHONE                     STRING,
    FIRSTSCANNEDDATE                TIMESTAMP,
    LASTPRINTEDDATE                 TIMESTAMP,
    ACCESSVALIDITYMODIFIEDDATE      TIMESTAMP,
    CREATEDDATE                     TIMESTAMP,
    COMPANYPRODUCTCODE              STRING,
    PAYMENTSTATUS                   STRING,
    PHOTOKEY                        STRING,
    PHOTOSOURCE                     STRING,
    PHOTOSOURCETYPE                 STRING
) USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {50}: Insert into staging table
-- Converted: ODI MAX self-join dedup replaced with ROW_NUMBER() OVER (PARTITION BY ID ORDER BY INT_INSERT_DATE DESC, VERSIONNUMBER DESC) (Rule F.12)
-- Converted: removed /*+ append */, TO_TIMESTAMP Oracle format → Spark format, schema → workspace
INSERT INTO workspace.prxbi_dw.c_0a7sucripsm1cg2656h955ou5qp_stg
(
    ID,
    BADGELOCATION,
    BADGETOKEN,
    BADGEVERSION,
    CONTACTEMAIL,
    CONTACTFIRSTNAME,
    CONTACTJOBTITLE,
    CONTACTLASTNAME,
    CONTACTPERSONRXMASTERID,
    CREATEDBYREGISTRATIONTYPE,
    CREATEDBYTYPE,
    CULTURE,
    CUSTOMERTYPE,
    EVENTEDITIONGBSCODE,
    ISBADGEUPDATE,
    MARKETINGPREFERENCESPROMPTREQU,
    ORGANISATIONCITY,
    ORGANISATIONCOUNTRYCODE,
    ORGANISATIONDISPLAYNAME,
    ORGANISATIONRXMASTERID,
    ORGANISATIONSTATE,
    PARTICIPATINGORGANISATIONID,
    PRODUCTCODE,
    QRCODECONTENT,
    REGISTRATIONID,
    STATUS,
    SUPPORTSTAFFCOMPANYADDRESS,
    SUPPORTSTAFFCOMPANYNAME,
    SUPPORTSTAFFMOBILEPHONE,
    SUPPORTSTAFFREPORTSTO,
    SUPPORTSTAFFSTANDS,
    SUPPORTSTAFFUSERACCESS,
    VERSIONNUMBER,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE
)
SELECT
    ID,
    BADGELOCATION,
    BADGETOKEN,
    BADGEVERSION,
    CONTACTEMAIL,
    CONTACTFIRSTNAME,
    CONTACTJOBTITLE,
    CONTACTLASTNAME,
    CONTACTPERSONRXMASTERID,
    CREATEDBYREGISTRATIONTYPE,
    CREATEDBYTYPE,
    CULTURE,
    CUSTOMERTYPE,
    EVENTEDITIONGBSCODE,
    ISBADGEUPDATE,
    MARKETINGPREFERENCESPROMPTREQUIRED AS MARKETINGPREFERENCESPROMPTREQU,
    ORGANISATIONCITY,
    ORGANISATIONCOUNTRYCODE,
    ORGANISATIONDISPLAYNAME,
    ORGANISATIONRXMASTERID,
    ORGANISATIONSTATE,
    PARTICIPATINGORGANISATIONID,
    PRODUCTCODE,
    QRCODECONTENT,
    REGISTRATIONID,
    STATUS,
    SUPPORTSTAFFCOMPANYADDRESS,
    SUPPORTSTAFFCOMPANYNAME,
    SUPPORTSTAFFMOBILEPHONE,
    SUPPORTSTAFFREPORTSTO,
    SUPPORTSTAFFSTANDS,
    SUPPORTSTAFFUSERACCESS,
    VERSIONNUMBER,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE
FROM (
    SELECT
        ID,
        BADGELOCATION,
        BADGETOKEN,
        BADGEVERSION,
        CONTACTEMAIL,
        CONTACTFIRSTNAME,
        CONTACTJOBTITLE,
        CONTACTLASTNAME,
        CONTACTPERSONRXMASTERID,
        CREATEDBYREGISTRATIONTYPE,
        CREATEDBYTYPE,
        CULTURE,
        CUSTOMERTYPE,
        EVENTEDITIONGBSCODE,
        ISBADGEUPDATE,
        MARKETINGPREFERENCESPROMPTREQUIRED,
        ORGANISATIONCITY,
        ORGANISATIONCOUNTRYCODE,
        ORGANISATIONDISPLAYNAME,
        ORGANISATIONRXMASTERID,
        ORGANISATIONSTATE,
        PARTICIPATINGORGANISATIONID,
        PRODUCTCODE,
        QRCODECONTENT,
        REGISTRATIONID,
        STATUS,
        SUPPORTSTAFFCOMPANYADDRESS,
        SUPPORTSTAFFCOMPANYNAME,
        SUPPORTSTAFFMOBILEPHONE,
        SUPPORTSTAFFREPORTSTO,
        SUPPORTSTAFFSTANDS,
        SUPPORTSTAFFUSERACCESS,
        VERSIONNUMBER,
        MOBILEPHONE,
        FIRSTSCANNEDDATE,
        LASTPRINTEDDATE,
        ACCESSVALIDITYMODIFIEDDATE,
        CREATEDDATE,
        COMPANYPRODUCTCODE,
        PAYMENTSTATUS,
        PHOTOKEY,
        PHOTOSOURCE,
        PHOTOSOURCETYPE,
        ROW_NUMBER() OVER (
            PARTITION BY ID
            ORDER BY INT_INSERT_DATE DESC, VERSIONNUMBER DESC
        ) AS rn
    FROM workspace.prxbi_ts.wc_mercury_badge_ts
    WHERE INT_INSERT_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
      AND INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
) deduped
WHERE rn = 1;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {60}: Staging record count (replaces DBMS_STATS)
SELECT COUNT(*) AS staging_row_count
FROM workspace.prxbi_dw.c_0a7sucripsm1cg2656h955ou5qp_stg;

## Flow Table (I$)
Drop, create, and populate the flow/integration table with dimension lookups and NOT_EXISTS detection.

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {80}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {90}: Create flow table
-- Converted: VARCHAR2 → STRING, NUMBER(10,0)/NUMBER(5,0) → BIGINT, DATE → TIMESTAMP,
--            TIMESTAMP(6)/TIMESTAMP(7) → TIMESTAMP, CHAR(1) → STRING, removed NOLOGGING
CREATE TABLE workspace.prxbi_dw.i_wc_badge_details_d_flow (
    ROW_WID                     BIGINT,
    BADGE_ID                    STRING,
    BADGE_LOCATION              STRING,
    BADGE_TOKEN                 STRING,
    BADGE_VERSION               BIGINT,
    CONTACT_EMAIL               STRING,
    CONTACT_FIRST_NAME          STRING,
    CONTACT_LAST_NAME           STRING,
    CONTACT_JOB_TITLE           STRING,
    CONTACT_PERSON_ID           STRING,
    CREATION_REG_TYPE           STRING,
    CREATION_TYPE               STRING,
    CULTURE                     STRING,
    CUSTOMER_TYPE               STRING,
    EVENT_EDITION_CODE          STRING,
    BADGE_UPDATE_FLG            STRING,
    MARKETING_PREF_PROMPT       STRING,
    ORG_NAME                    STRING,
    ORG_CITY                    STRING,
    ORG_COUNTRY                 STRING,
    ORG_ID                      STRING,
    ORG_STATE                   STRING,
    PARTICIPATING_ORG_ID        STRING,
    PRODUCT_CODE                STRING,
    QR_CODE                     STRING,
    REGISTRATION_ID             STRING,
    STATUS                      BIGINT,
    STAFF_COMPANY_NAME          STRING,
    STAFF_COMPANY_ADDR          STRING,
    STAFF_PHONE_NUM             STRING,
    STAFF_REPORTING             STRING,
    STAFF_STANDS                STRING,
    STAFF_USER_ACCESS           STRING,
    VERSION_NUM                 BIGINT,
    INTEGRATION_ID              STRING,
    DATASOURCE_NUM_ID           STRING,
    W_INSERT_DT                 TIMESTAMP,
    W_UPDATE_DT                 TIMESTAMP,
    MOBILEPHONE                 STRING,
    FIRSTSCANNEDDATE            TIMESTAMP,
    LASTPRINTEDDATE             TIMESTAMP,
    FIRSTSCANNEDDATE_FLG        STRING,
    LASTPRINTEDDATE_FLG         STRING,
    ACCESSVALIDITYMODIFIEDDATE  TIMESTAMP,
    CREATEDDATE                 TIMESTAMP,
    COMPANYPRODUCTCODE          STRING,
    PAYMENTSTATUS               STRING,
    PHOTOKEY                    STRING,
    PHOTOSOURCE                 STRING,
    PHOTOSOURCETYPE             STRING,
    PACKAGE_NAME                STRING,
    IND_UPDATE                  STRING
) USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {100}:
Insert into flow table (DETECTION_STRATEGY = NOT_EXISTS)
-- Converted: removed /*+ append */, NVL2 → CASE WHEN, schema → workspace
-- NOT EXISTS correlated subquery is preserved here as it's in INSERT...SELECT
WHERE NOT EXISTS (allowed in Spark)
-- rank() window function preserved as-is (Spark compatible)
INSERT INTO workspace.prxbi_dw.i_wc_badge_details_d_flow
(
    BADGE_ID,
    BADGE_LOCATION,
    BADGE_TOKEN,
    BADGE_VERSION,
    CONTACT_EMAIL,
    CONTACT_FIRST_NAME,
    CONTACT_LAST_NAME,
    CONTACT_JOB_TITLE,
    CONTACT_PERSON_ID,
    CREATION_REG_TYPE,
    CREATION_TYPE,
    CULTURE,
    CUSTOMER_TYPE,
    EVENT_EDITION_CODE,
    BADGE_UPDATE_FLG,
    MARKETING_PREF_PROMPT,
    ORG_NAME,
    ORG_CITY,
    ORG_COUNTRY,
    ORG_ID,
    ORG_STATE,
    PARTICIPATING_ORG_ID,
    PRODUCT_CODE,
    QR_CODE,
    REGISTRATION_ID,
    STATUS,
    STAFF_COMPANY_NAME,
    STAFF_COMPANY_ADDR,
    STAFF_PHONE_NUM,
    STAFF_REPORTING,
    STAFF_STANDS,
    STAFF_USER_ACCESS,
    VERSION_NUM,
    INTEGRATION_ID,
    DATASOURCE_NUM_ID,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    FIRSTSCANNEDDATE_FLG,
    LASTPRINTEDDATE_FLG,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE,
    PACKAGE_NAME,
    IND_UPDATE
)
SELECT S.BADGE_ID,
    S.BADGE_LOCATION,
    S.BADGE_TOKEN,
    S.BADGE_VERSION,
    S.CONTACT_EMAIL,
    S.CONTACT_FIRST_NAME,
    S.CONTACT_LAST_NAME,
    S.CONTACT_JOB_TITLE,
    S.CONTACT_PERSON_ID,
    S.CREATION_REG_TYPE,
    S.CREATION_TYPE,
    S.CULTURE,
    S.CUSTOMER_TYPE,
    S.EVENT_EDITION_CODE,
    S.BADGE_UPDATE_FLG,
    S.MARKETING_PREF_PROMPT,
    S.ORG_NAME,
    S.ORG_CITY,
    S.ORG_COUNTRY,
    S.ORG_ID,
    S.ORG_STATE,
    S.PARTICIPATING_ORG_ID,
    S.PRODUCT_CODE,
    S.QR_CODE,
    S.REGISTRATION_ID,
    S.STATUS,
    S.STAFF_COMPANY_NAME,
    S.STAFF_COMPANY_ADDR,
    S.STAFF_PHONE_NUM,
    S.STAFF_REPORTING,
    S.STAFF_STANDS,
    S.STAFF_USER_ACCESS,
    S.VERSION_NUM,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.MOBILEPHONE,
    S.FIRSTSCANNEDDATE,
    S.LASTPRINTEDDATE,
    S.FIRSTSCANNEDDATE_FLG,
    S.LASTPRINTEDDATE_FLG,
    S.ACCESSVALIDITYMODIFIEDDATE,
    S.CREATEDDATE,
    S.COMPANYPRODUCTCODE,
    S.PAYMENTSTATUS,
    S.PHOTOKEY,
    S.PHOTOSOURCE,
    S.PHOTOSOURCETYPE,
    S.PACKAGE_NAME,
    S.IND_UPDATE
FROM (
SELECT JOIN1_A.ID                              AS BADGE_ID,
        JOIN1_A.BADGELOCATION                   AS BADGE_LOCATION,
        JOIN1_A.BADGETOKEN                      AS BADGE_TOKEN,
        JOIN1_A.BADGEVERSION                    AS BADGE_VERSION,
        JOIN1_A.CONTACTEMAIL                    AS CONTACT_EMAIL,
        JOIN1_A.CONTACTFIRSTNAME                AS CONTACT_FIRST_NAME,
        JOIN1_A.CONTACTLASTNAME                 AS CONTACT_LAST_NAME,
        JOIN1_A.CONTACTJOBTITLE                 AS CONTACT_JOB_TITLE,
        JOIN1_A.CONTACTPERSONRXMASTERID         AS CONTACT_PERSON_ID,
        JOIN1_A.CREATEDBYREGISTRATIONTYPE       AS CREATION_REG_TYPE,
        JOIN1_A.CREATEDBYTYPE                   AS CREATION_TYPE,
        JOIN1_A.CULTURE                         AS CULTURE,
        JOIN1_A.CUSTOMERTYPE                    AS CUSTOMER_TYPE,
        JOIN1_A.EVENTEDITIONGBSCODE             AS EVENT_EDITION_CODE,
        JOIN1_A.ISBADGEUPDATE                   AS BADGE_UPDATE_FLG,
        JOIN1_A.MARKETINGPREFERENCESPROMPTREQU  AS MARKETING_PREF_PROMPT,
        JOIN1_A.ORGANISATIONDISPLAYNAME         AS ORG_NAME,
        JOIN1_A.ORGANISATIONCITY                AS ORG_CITY,
        JOIN1_A.ORGANISATIONCOUNTRYCODE         AS ORG_COUNTRY,
        JOIN1_A.ORGANISATIONRXMASTERID          AS ORG_ID,
        JOIN1_A.ORGANISATIONSTATE               AS ORG_STATE,
        JOIN1_A.PARTICIPATINGORGANISATIONID     AS PARTICIPATING_ORG_ID,
        JOIN1_A.PRODUCTCODE                     AS PRODUCT_CODE,
        JOIN1_A.QRCODECONTENT                   AS QR_CODE,
        JOIN1_A.REGISTRATIONID                  AS REGISTRATION_ID,
        JOIN1_A.STATUS                          AS STATUS,
        JOIN1_A.SUPPORTSTAFFCOMPANYNAME         AS STAFF_COMPANY_NAME,
        JOIN1_A.SUPPORTSTAFFCOMPANYADDRESS      AS STAFF_COMPANY_ADDR,
        JOIN1_A.SUPPORTSTAFFMOBILEPHONE         AS STAFF_PHONE_NUM,
        JOIN1_A.SUPPORTSTAFFREPORTSTO           AS STAFF_REPORTING,
        JOIN1_A.SUPPORTSTAFFSTANDS              AS STAFF_STANDS,
        JOIN1_A.SUPPORTSTAFFUSERACCESS          AS STAFF_USER_ACCESS,
        JOIN1_A.VERSIONNUMBER                   AS VERSION_NUM,
        JOIN1_A.ID                              AS INTEGRATION_ID,
        CAST(380 AS STRING)                     AS DATASOURCE_NUM_ID,
        JOIN1_A.MOBILEPHONE                     AS MOBILEPHONE,
        JOIN1_A.FIRSTSCANNEDDATE                AS FIRSTSCANNEDDATE,
        JOIN1_A.LASTPRINTEDDATE                 AS LASTPRINTEDDATE,
        CASE
WHEN JOIN1_A.FIRSTSCANNEDDATE IS NOT NULL
THEN 'Y' ELSE 'N' END AS FIRSTSCANNEDDATE_FLG,
        CASE
WHEN JOIN1_A.LASTPRINTEDDATE  IS NOT NULL
THEN 'Y' ELSE 'N' END AS LASTPRINTEDDATE_FLG,
        JOIN1_A.ACCESSVALIDITYMODIFIEDDATE      AS ACCESSVALIDITYMODIFIEDDATE,
        JOIN1_A.CREATEDDATE                     AS CREATEDDATE,
        JOIN1_A.COMPANYPRODUCTCODE              AS COMPANYPRODUCTCODE,
        JOIN1_A.PAYMENTSTATUS                   AS PAYMENTSTATUS,
        JOIN1_A.PHOTOKEY                        AS PHOTOKEY,
        JOIN1_A.PHOTOSOURCE                     AS PHOTOSOURCE,
        JOIN1_A.PHOTOSOURCETYPE                 AS PHOTOSOURCETYPE,
        WC_BADGE_PRODUCT_D_2.NAME_1             AS PACKAGE_NAME,
        'I'                                     AS IND_UPDATE
FROM workspace.prxbi_dw.c_0a7sucripsm1cg2656h955ou5qp_stg JOIN1_A
LEFT OUTER
JOIN (
SELECT WC_BADGE_PRODUCT_D_1.ID     AS ID,
            WC_BADGE_PRODUCT_D_1.SKU    AS SKU,
            WC_BADGE_PRODUCT_D_1.NAME   AS NAME,
            WC_BADGE_PRODUCT_D_1.COL    AS COL,
            WC_BADGE_PRODUCT_D_1.SKU_1  AS SKU_1,
            WC_BADGE_PRODUCT_D_1.NAME_1 AS NAME_1
FROM (
SELECT WC_BADGE_PRODUCT_D.ID   AS ID,
                WC_BADGE_PRODUCT_D.SKU  AS SKU,
                WC_BADGE_PRODUCT_D.NAME AS NAME,
                rank() OVER (PARTITION BY WC_BADGE_PRODUCT_D.SKU
ORDER BY WC_BADGE_PRODUCT_D.ID DESC) AS COL,
                WC_BADGE_PRODUCT_D.SKU  AS SKU_1,
                WC_BADGE_PRODUCT_D.NAME AS NAME_1
FROM workspace.prxbi_dw.wc_badge_product_d WC_BADGE_PRODUCT_D
        ) WC_BADGE_PRODUCT_D_1
WHERE WC_BADGE_PRODUCT_D_1.COL = 1
    ) WC_BADGE_PRODUCT_D_2
ON JOIN1_A.PRODUCTCODE = WC_BADGE_PRODUCT_D_2.SKU_1
) S
WHERE NOT EXISTS (
SELECT 1
FROM workspace.prxbi_dw.wc_badge_details_d T
WHERE T.INTEGRATION_ID     = S.INTEGRATION_ID
AND T.DATASOURCE_NUM_ID  = S.DATASOURCE_NUM_ID
AND ((T.BADGE_ID = S.BADGE_ID)
OR (T.BADGE_ID IS NULL
AND S.BADGE_ID IS NULL))
AND ((T.BADGE_LOCATION = S.BADGE_LOCATION)
OR (T.BADGE_LOCATION IS NULL
AND S.BADGE_LOCATION IS NULL))
AND ((T.BADGE_TOKEN = S.BADGE_TOKEN)
OR (T.BADGE_TOKEN IS NULL
AND S.BADGE_TOKEN IS NULL))
AND ((T.BADGE_VERSION = S.BADGE_VERSION)
OR (T.BADGE_VERSION IS NULL
AND S.BADGE_VERSION IS NULL))
AND ((T.CONTACT_EMAIL = S.CONTACT_EMAIL)
OR (T.CONTACT_EMAIL IS NULL
AND S.CONTACT_EMAIL IS NULL))
AND ((T.CONTACT_FIRST_NAME = S.CONTACT_FIRST_NAME)
OR (T.CONTACT_FIRST_NAME IS NULL
AND S.CONTACT_FIRST_NAME IS NULL))
AND ((T.CONTACT_LAST_NAME = S.CONTACT_LAST_NAME)
OR (T.CONTACT_LAST_NAME IS NULL
AND S.CONTACT_LAST_NAME IS NULL))
AND ((T.CONTACT_JOB_TITLE = S.CONTACT_JOB_TITLE)
OR (T.CONTACT_JOB_TITLE IS NULL
AND S.CONTACT_JOB_TITLE IS NULL))
AND ((T.CONTACT_PERSON_ID = S.CONTACT_PERSON_ID)
OR (T.CONTACT_PERSON_ID IS NULL
AND S.CONTACT_PERSON_ID IS NULL))
AND ((T.CREATION_REG_TYPE = S.CREATION_REG_TYPE)
OR (T.CREATION_REG_TYPE IS NULL
AND S.CREATION_REG_TYPE IS NULL))
AND ((T.CREATION_TYPE = S.CREATION_TYPE)
OR (T.CREATION_TYPE IS NULL
AND S.CREATION_TYPE IS NULL))
AND ((T.CULTURE = S.CULTURE)
OR (T.CULTURE IS NULL
AND S.CULTURE IS NULL))
AND ((T.CUSTOMER_TYPE = S.CUSTOMER_TYPE)
OR (T.CUSTOMER_TYPE IS NULL
AND S.CUSTOMER_TYPE IS NULL))
AND ((T.EVENT_EDITION_CODE = S.EVENT_EDITION_CODE)
OR (T.EVENT_EDITION_CODE IS NULL
AND S.EVENT_EDITION_CODE IS NULL))
AND ((T.BADGE_UPDATE_FLG = S.BADGE_UPDATE_FLG)
OR (T.BADGE_UPDATE_FLG IS NULL
AND S.BADGE_UPDATE_FLG IS NULL))
AND ((T.MARKETING_PREF_PROMPT = S.MARKETING_PREF_PROMPT)
OR (T.MARKETING_PREF_PROMPT IS NULL
AND S.MARKETING_PREF_PROMPT IS NULL))
AND ((T.ORG_NAME = S.ORG_NAME)
OR (T.ORG_NAME IS NULL
AND S.ORG_NAME IS NULL))
AND ((T.ORG_CITY = S.ORG_CITY)
OR (T.ORG_CITY IS NULL
AND S.ORG_CITY IS NULL))
AND ((T.ORG_COUNTRY = S.ORG_COUNTRY)
OR (T.ORG_COUNTRY IS NULL
AND S.ORG_COUNTRY IS NULL))
AND ((T.ORG_ID = S.ORG_ID)
OR (T.ORG_ID IS NULL
AND S.ORG_ID IS NULL))
AND ((T.ORG_STATE = S.ORG_STATE)
OR (T.ORG_STATE IS NULL
AND S.ORG_STATE IS NULL))
AND ((T.PARTICIPATING_ORG_ID = S.PARTICIPATING_ORG_ID)
OR (T.PARTICIPATING_ORG_ID IS NULL
AND S.PARTICIPATING_ORG_ID IS NULL))
AND ((T.PRODUCT_CODE = S.PRODUCT_CODE)
OR (T.PRODUCT_CODE IS NULL
AND S.PRODUCT_CODE IS NULL))
AND ((T.QR_CODE = S.QR_CODE)
OR (T.QR_CODE IS NULL
AND S.QR_CODE IS NULL))
AND ((T.REGISTRATION_ID = S.REGISTRATION_ID)
OR (T.REGISTRATION_ID IS NULL
AND S.REGISTRATION_ID IS NULL))
AND ((T.STATUS = S.STATUS)
OR (T.STATUS IS NULL
AND S.STATUS IS NULL))
AND ((T.STAFF_COMPANY_NAME = S.STAFF_COMPANY_NAME)
OR (T.STAFF_COMPANY_NAME IS NULL
AND S.STAFF_COMPANY_NAME IS NULL))
AND ((T.STAFF_COMPANY_ADDR = S.STAFF_COMPANY_ADDR)
OR (T.STAFF_COMPANY_ADDR IS NULL
AND S.STAFF_COMPANY_ADDR IS NULL))
AND ((T.STAFF_PHONE_NUM = S.STAFF_PHONE_NUM)
OR (T.STAFF_PHONE_NUM IS NULL
AND S.STAFF_PHONE_NUM IS NULL))
AND ((T.STAFF_REPORTING = S.STAFF_REPORTING)
OR (T.STAFF_REPORTING IS NULL
AND S.STAFF_REPORTING IS NULL))
AND ((T.STAFF_STANDS = S.STAFF_STANDS)
OR (T.STAFF_STANDS IS NULL
AND S.STAFF_STANDS IS NULL))
AND ((T.STAFF_USER_ACCESS = S.STAFF_USER_ACCESS)
OR (T.STAFF_USER_ACCESS IS NULL
AND S.STAFF_USER_ACCESS IS NULL))
AND ((T.VERSION_NUM = S.VERSION_NUM)
OR (T.VERSION_NUM IS NULL
AND S.VERSION_NUM IS NULL))
AND ((T.MOBILEPHONE = S.MOBILEPHONE)
OR (T.MOBILEPHONE IS NULL
AND S.MOBILEPHONE IS NULL))
AND ((T.FIRSTSCANNEDDATE = S.FIRSTSCANNEDDATE)
OR (T.FIRSTSCANNEDDATE IS NULL
AND S.FIRSTSCANNEDDATE IS NULL))
AND ((T.LASTPRINTEDDATE = S.LASTPRINTEDDATE)
OR (T.LASTPRINTEDDATE IS NULL
AND S.LASTPRINTEDDATE IS NULL))
AND ((T.FIRSTSCANNEDDATE_FLG = S.FIRSTSCANNEDDATE_FLG)
OR (T.FIRSTSCANNEDDATE_FLG IS NULL
AND S.FIRSTSCANNEDDATE_FLG IS NULL))
AND ((T.LASTPRINTEDDATE_FLG = S.LASTPRINTEDDATE_FLG)
OR (T.LASTPRINTEDDATE_FLG IS NULL
AND S.LASTPRINTEDDATE_FLG IS NULL))
AND ((T.ACCESSVALIDITYMODIFIEDDATE = S.ACCESSVALIDITYMODIFIEDDATE)
OR (T.ACCESSVALIDITYMODIFIEDDATE IS NULL
AND S.ACCESSVALIDITYMODIFIEDDATE IS NULL))
AND ((T.CREATEDDATE = S.CREATEDDATE)
OR (T.CREATEDDATE IS NULL
AND S.CREATEDDATE IS NULL))
AND ((T.COMPANYPRODUCTCODE = S.COMPANYPRODUCTCODE)
OR (T.COMPANYPRODUCTCODE IS NULL
AND S.COMPANYPRODUCTCODE IS NULL))
AND ((T.PAYMENTSTATUS = S.PAYMENTSTATUS)
OR (T.PAYMENTSTATUS IS NULL
AND S.PAYMENTSTATUS IS NULL))
AND ((T.PHOTOKEY = S.PHOTOKEY)
OR (T.PHOTOKEY IS NULL
AND S.PHOTOKEY IS NULL))
AND ((T.PHOTOSOURCE = S.PHOTOSOURCE)
OR (T.PHOTOSOURCE IS NULL
AND S.PHOTOSOURCE IS NULL))
AND ((T.PHOTOSOURCETYPE = S.PHOTOSOURCETYPE)
OR (T.PHOTOSOURCETYPE IS NULL
AND S.PHOTOSOURCETYPE IS NULL))
AND ((T.PACKAGE_NAME = S.PACKAGE_NAME)
OR (T.PACKAGE_NAME IS NULL
AND S.PACKAGE_NAME IS NULL))
);

In [ ]:
-- MAGIC %sql
-- Flow table record count
SELECT COUNT(*) AS flow_row_count
FROM workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {110}:
Optimize flow table (replaces
CREATE INDEX + DBMS_STATS)
-- Disable
ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.i_wc_badge_details_d_flow
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

## Mark Records for Update (IND_UPDATE)
Flag existing records in the flow table as 'U' where they already exist in the target.
Converted from Oracle tuple-IN UPDATE (Rule F.10) → MERGE (Rule 7.3).

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {130}: Mark IND_UPDATE = 'U' for existing records
-- Converted: WHERE (INTEGRATION_ID, DATASOURCE_NUM_ID) IN (SELECT ...) → MERGE (Rule F.10 / Rule 7.3)
MERGE INTO workspace.prxbi_dw.i_wc_badge_details_d_flow AS T
USING (
    SELECT INTEGRATION_ID, DATASOURCE_NUM_ID
    FROM workspace.prxbi_dw.wc_badge_details_d
) AS S
ON T.INTEGRATION_ID    = S.INTEGRATION_ID
AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

## MERGE into Target Table
Combines SCEN_TASK_NO {150} (Oracle tuple-SET UPDATE) and {160} (INSERT where IND_UPDATE='I')
into a single MERGE statement.
Converted: Oracle tuple-SET UPDATE → MERGE (Rule F.3), SEQUENCE.NEXTVAL removed (ROW_WID is GENERATED ALWAYS AS IDENTITY, Rule F.4), SYSTIMESTAMP → current_timestamp().

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {150} + {160}: MERGE into target WC_BADGE_DETAILS_D
-- Converted: Oracle tuple-SET UPDATE → MERGE WHEN MATCHED UPDATE SET (Rule F.3)
-- Converted: INSERT with SEQUENCE.NEXTVAL → MERGE WHEN NOT MATCHED INSERT (ROW_WID excluded, Rule F.4)
-- Converted: SYSTIMESTAMP → current_timestamp()
-- NOTE: ROW_WID must be defined as GENERATED ALWAYS AS IDENTITY on the target table;
--       it is excluded from all INSERT and UPDATE column lists per Rule F.4.
MERGE INTO workspace.prxbi_dw.wc_badge_details_d AS T
USING workspace.prxbi_dw.i_wc_badge_details_d_flow AS S
ON T.INTEGRATION_ID    = S.INTEGRATION_ID
AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.BADGE_ID                   = S.BADGE_ID,
    T.BADGE_LOCATION             = S.BADGE_LOCATION,
    T.BADGE_TOKEN                = S.BADGE_TOKEN,
    T.BADGE_VERSION              = S.BADGE_VERSION,
    T.CONTACT_EMAIL              = S.CONTACT_EMAIL,
    T.CONTACT_FIRST_NAME         = S.CONTACT_FIRST_NAME,
    T.CONTACT_LAST_NAME          = S.CONTACT_LAST_NAME,
    T.CONTACT_JOB_TITLE          = S.CONTACT_JOB_TITLE,
    T.CONTACT_PERSON_ID          = S.CONTACT_PERSON_ID,
    T.CREATION_REG_TYPE          = S.CREATION_REG_TYPE,
    T.CREATION_TYPE              = S.CREATION_TYPE,
    T.CULTURE                    = S.CULTURE,
    T.CUSTOMER_TYPE              = S.CUSTOMER_TYPE,
    T.EVENT_EDITION_CODE         = S.EVENT_EDITION_CODE,
    T.BADGE_UPDATE_FLG           = S.BADGE_UPDATE_FLG,
    T.MARKETING_PREF_PROMPT      = S.MARKETING_PREF_PROMPT,
    T.ORG_NAME                   = S.ORG_NAME,
    T.ORG_CITY                   = S.ORG_CITY,
    T.ORG_COUNTRY                = S.ORG_COUNTRY,
    T.ORG_ID                     = S.ORG_ID,
    T.ORG_STATE                  = S.ORG_STATE,
    T.PARTICIPATING_ORG_ID       = S.PARTICIPATING_ORG_ID,
    T.PRODUCT_CODE               = S.PRODUCT_CODE,
    T.QR_CODE                    = S.QR_CODE,
    T.REGISTRATION_ID            = S.REGISTRATION_ID,
    T.STATUS                     = S.STATUS,
    T.STAFF_COMPANY_NAME         = S.STAFF_COMPANY_NAME,
    T.STAFF_COMPANY_ADDR         = S.STAFF_COMPANY_ADDR,
    T.STAFF_PHONE_NUM            = S.STAFF_PHONE_NUM,
    T.STAFF_REPORTING            = S.STAFF_REPORTING,
    T.STAFF_STANDS               = S.STAFF_STANDS,
    T.STAFF_USER_ACCESS          = S.STAFF_USER_ACCESS,
    T.VERSION_NUM                = S.VERSION_NUM,
    T.MOBILEPHONE                = S.MOBILEPHONE,
    T.FIRSTSCANNEDDATE           = S.FIRSTSCANNEDDATE,
    T.LASTPRINTEDDATE            = S.LASTPRINTEDDATE,
    T.FIRSTSCANNEDDATE_FLG       = S.FIRSTSCANNEDDATE_FLG,
    T.LASTPRINTEDDATE_FLG        = S.LASTPRINTEDDATE_FLG,
    T.ACCESSVALIDITYMODIFIEDDATE = S.ACCESSVALIDITYMODIFIEDDATE,
    T.CREATEDDATE                = S.CREATEDDATE,
    T.COMPANYPRODUCTCODE         = S.COMPANYPRODUCTCODE,
    T.PAYMENTSTATUS              = S.PAYMENTSTATUS,
    T.PHOTOKEY                   = S.PHOTOKEY,
    T.PHOTOSOURCE                = S.PHOTOSOURCE,
    T.PHOTOSOURCETYPE            = S.PHOTOSOURCETYPE,
    T.PACKAGE_NAME               = S.PACKAGE_NAME,
    T.W_UPDATE_DT                = current_timestamp()
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
    BADGE_ID,
    BADGE_LOCATION,
    BADGE_TOKEN,
    BADGE_VERSION,
    CONTACT_EMAIL,
    CONTACT_FIRST_NAME,
    CONTACT_LAST_NAME,
    CONTACT_JOB_TITLE,
    CONTACT_PERSON_ID,
    CREATION_REG_TYPE,
    CREATION_TYPE,
    CULTURE,
    CUSTOMER_TYPE,
    EVENT_EDITION_CODE,
    BADGE_UPDATE_FLG,
    MARKETING_PREF_PROMPT,
    ORG_NAME,
    ORG_CITY,
    ORG_COUNTRY,
    ORG_ID,
    ORG_STATE,
    PARTICIPATING_ORG_ID,
    PRODUCT_CODE,
    QR_CODE,
    REGISTRATION_ID,
    STATUS,
    STAFF_COMPANY_NAME,
    STAFF_COMPANY_ADDR,
    STAFF_PHONE_NUM,
    STAFF_REPORTING,
    STAFF_STANDS,
    STAFF_USER_ACCESS,
    VERSION_NUM,
    INTEGRATION_ID,
    DATASOURCE_NUM_ID,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    FIRSTSCANNEDDATE_FLG,
    LASTPRINTEDDATE_FLG,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE,
    PACKAGE_NAME,
    W_INSERT_DT,
    W_UPDATE_DT
) VALUES (
    S.BADGE_ID,
    S.BADGE_LOCATION,
    S.BADGE_TOKEN,
    S.BADGE_VERSION,
    S.CONTACT_EMAIL,
    S.CONTACT_FIRST_NAME,
    S.CONTACT_LAST_NAME,
    S.CONTACT_JOB_TITLE,
    S.CONTACT_PERSON_ID,
    S.CREATION_REG_TYPE,
    S.CREATION_TYPE,
    S.CULTURE,
    S.CUSTOMER_TYPE,
    S.EVENT_EDITION_CODE,
    S.BADGE_UPDATE_FLG,
    S.MARKETING_PREF_PROMPT,
    S.ORG_NAME,
    S.ORG_CITY,
    S.ORG_COUNTRY,
    S.ORG_ID,
    S.ORG_STATE,
    S.PARTICIPATING_ORG_ID,
    S.PRODUCT_CODE,
    S.QR_CODE,
    S.REGISTRATION_ID,
    S.STATUS,
    S.STAFF_COMPANY_NAME,
    S.STAFF_COMPANY_ADDR,
    S.STAFF_PHONE_NUM,
    S.STAFF_REPORTING,
    S.STAFF_STANDS,
    S.STAFF_USER_ACCESS,
    S.VERSION_NUM,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.MOBILEPHONE,
    S.FIRSTSCANNEDDATE,
    S.LASTPRINTEDDATE,
    S.FIRSTSCANNEDDATE_FLG,
    S.LASTPRINTEDDATE_FLG,
    S.ACCESSVALIDITYMODIFIEDDATE,
    S.CREATEDDATE,
    S.COMPANYPRODUCTCODE,
    S.PAYMENTSTATUS,
    S.PHOTOKEY,
    S.PHOTOSOURCE,
    S.PHOTOSOURCETYPE,
    S.PACKAGE_NAME,
    current_timestamp(),
    current_timestamp()
);

## Optimize Target Table

In [ ]:
-- MAGIC %sql
--
Optimize target table after
MERGE -- Disable
ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.wc_badge_details_d
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

## Cleanup
Drop temporary staging and flow tables. SCEN_TASK_NO {170} commit is implicit in Delta.

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {180}:
Drop flow table
-- SCEN_TASK_NO {170}: /*commit*/ — implicit in Databricks Delta, no action required
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {200} / {210}:
Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_0a7sucripsm1cg2656h955ou5qp_stg;

## Validation

In [ ]:
-- MAGIC %sql
-- Final record count in target
SELECT COUNT(*) AS target_row_count
FROM workspace.prxbi_dw.wc_badge_details_d
WHERE DATASOURCE_NUM_ID = '${DATASOURCE_NUM_ID}';

In [ ]:
-- MAGIC %sql
-- Sample recently inserted or updated records
SELECT
    INTEGRATION_ID,
    DATASOURCE_NUM_ID,
    BADGE_ID,
    EVENT_EDITION_CODE,
    W_INSERT_DT,
    W_UPDATE_DT
FROM workspace.prxbi_dw.wc_badge_details_d
WHERE DATASOURCE_NUM_ID = '${DATASOURCE_NUM_ID}'
ORDER BY W_UPDATE_DT DESC
LIMIT 20;

## Conversion Notes and Manual Actions Required

### Automated Conversions Applied
| # | Oracle | Spark | Rule |
|---|--------|-------|------|
| 1 | `VARCHAR2(n CHAR)` | `STRING` | Section 5 |
| 2 | `NUMBER(5,0)` / `NUMBER(10,0)` | `BIGINT` | Section 5 |
| 3 | `TIMESTAMP(6)` / `TIMESTAMP(7)` | `TIMESTAMP` | Section 5 |
| 4 | `DATE` (W_INSERT_DT, W_UPDATE_DT) | `TIMESTAMP` | Section 5 |
| 5 | `CHAR(1)` | `STRING` | Section 5 |
| 6 | `NOLOGGING` | removed | Rule 4.25 |
| 7 | `/*+ append */` | removed | Rule 4.24 |
| 8 | `DROP TABLE ... PURGE` | `DROP TABLE IF EXISTS` | Rule 4.26 |
| 9 | `BEGIN DBMS_STATS ... END` | `OPTIMIZE ... ZORDER BY` | Rule 4.23 |
| 10 | `CREATE INDEX ... NOLOGGING` | `OPTIMIZE ... ZORDER BY` | Rule 4.22 |
| 11 | `NVL2(col, 'Y', 'N')` | `CASE WHEN col IS NOT NULL THEN 'Y' ELSE 'N' END` | Rule 4.2 |
| 12 | `SYSTIMESTAMP` | `current_timestamp()` | Rule 4.5 |
| 13 | `SEQUENCE.NEXTVAL` (ROW_WID) | excluded from MERGE (GENERATED ALWAYS AS IDENTITY) | Rule F.4 |
| 14 | Oracle tuple-SET UPDATE (SCEN_TASK_NO {150}) | MERGE WHEN MATCHED UPDATE SET | Rule F.3 |
| 15 | Oracle INSERT ... WHERE NOT EXISTS (SCEN_TASK_NO {160}) | MERGE WHEN NOT MATCHED INSERT | Section 7.1 |
| 16 | `WHERE (INTEGRATION_ID, DATASOURCE_NUM_ID) IN (SELECT ...)` (SCEN_TASK_NO {130}) | MERGE with individual ON conditions | Rule F.10 / 7.3 |
| 17 | ODI MAX self-join dedup (SCEN_TASK_NO {50}) | `ROW_NUMBER() OVER (PARTITION BY ID ORDER BY INT_INSERT_DATE DESC, VERSIONNUMBER DESC)` | Rule F.12 |
| 18 | `PRXBI_DW_SEP.*` / `PRXBI_TS_SEP.*` | `workspace.prxbi_dw.*` / `workspace.prxbi_ts.*` | Section 8.1 |
| 19 | `#GLOBAL.v_ETL_JOB_TYPE` | `'${ETL_JOB_TYPE}'` | Section 8.3 |
| 20 | `/*commit*/` (SCEN_TASK_NO {170}) | removed — implicit in Delta | Rule 4.27 |

### Manual Actions Required Before Running
1. **ROW_WID on WC_BADGE_DETAILS_D**: Ensure the target table `workspace.prxbi_dw.wc_badge_details_d` has `ROW_WID` defined as `BIGINT GENERATED ALWAYS AS IDENTITY`. The original ODI INSERT used `WC_BADGE_DETAILS_D_SEQ.NEXTVAL`. This column has been excluded from the MERGE per Rule F.4.
2. **wc_etl_parameters table**: Confirm `workspace.prxbi_dw.wc_etl_parameters` exists with columns `ETL_JOB_TYPE`, `etl_last_extract_time`, `etl_current_extract_time`, `ROW_WID`.
3. **wc_badge_product_d table**: Confirm `workspace.prxbi_dw.wc_badge_product_d` exists with columns `ID`, `SKU`, `NAME`.
4. **wc_mercury_badge_ts source table**: Confirm `workspace.prxbi_ts.wc_mercury_badge_ts` exists with column `INT_INSERT_DATE` for incremental extraction and `MARKETINGPREFERENCESPROMPTREQUIRED` (full name — the staging table truncates this to 30 chars in the ODI column alias).
5. **SCEN_TASK_NO {140}**: This step was skipped in ODI (`Command skipped due to chosen DETECTION_STRATEGY`) — no conversion needed.
6. **Widget defaults**: Review default values for `ETL_JOB_TYPE` and `DATASOURCE_NUM_ID` widgets before production execution.